# POSEIDON: espectros de **tránsito** con contaminación estelar y **química variable**
**Objetivo.** Este cuaderno genera espectros de transmisión con POSEIDON incluyendo **manchas/fáculas no ocultadas** y te deja **variar fácilmente la química** (lista de especies y sus mezclas). También agrega una **plantilla** para correr un retrieval (opcional).

> **Nota:** Este cuaderno está organizado en bloques independientes. Puedes simular primero y, si lo deseas, luego activar la celda de _retrieval_. 

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 1) Instalación (descomenta si lo necesitas, p. ej. en Google Colab)
# ──────────────────────────────────────────────────────────────────────────────
# %pip install -q poseidon-retrievals pysynphot msg
# %pip install -q numpy scipy matplotlib pandas
#
# (Opcional) backends de muestreo para retrievals avanzados:
# %pip install -q pymultinest dynesty emcee
#
# Si usas PyMultiNest, necesitarás libmultinest en el sistema.
# En Colab suele ser más simple usar 'dynesty' o 'emcee'.


In [1]:
# ──────────────────────────────────────────────────────────────────────────────
# 2) Imports
# ──────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Núcleo de POSEIDON
from POSEIDON.core import (
    create_star, create_planet, define_model,
    wl_grid_constant_R, read_opacities, make_atmosphere, compute_spectrum,
    load_data, set_priors
)
from POSEIDON.stellar import stellar_contamination, precompute_stellar_spectra

# Para reproducibilidad
rng = np.random.default_rng(42)


/home/dasan/anaconda3/envs/POSEIDON/lib/python3.10/site-packages/pysynphot/__init__.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


## Parámetros rápidos (ajústalos y ejecuta todo)
- **Estrella**: temperatura efectiva, log g, metalicidad, y esquema de contaminación (`one_spot` o `two_spots`).
- **Heterogeneidades**: fracciones y temperaturas de **manchas** y **fáculas**.
- **Planeta/Modelo**: lista de especies paramétricas (`param_species`) y a granel (`bulk_species`), así como el perfil (isotermo/isóquímico).
- **Rejilla espectral**: rango y resolución.


In [2]:
# ──────────────────────────────────────────────────────────────────────────────
# 3) KNOBS del usuario
# ──────────────────────────────────────────────────────────────────────────────

# --- Estrella y contaminación ---
star_kwargs = dict(
    R_s = 0.121*6.957e8,    # Radio estelar (m); ej. TRAPPIST-1 ~0.121 R_sun
    T_eff = 2550.0,         # K
    log_g = 5.2,            # log10(cm/s^2)
    Met   = 0.0,            # [Fe/H]
    stellar_grid = 'cbk04', # 'cbk04' o 'phoenix' (vía pysynphot), o 'Goettingen-HiRes' (pymsg)
    # Esquema: 'one_spot', 'one_spot_free_log_g', 'two_spots' o 'two_spots_free_log_g'
    stellar_contam = 'two_spots',
    # --- Para two_spots ---
    f_spot = 0.03,          # fracción de manchas
    f_fac  = 0.02,          # fracción de fáculas
    T_spot = 2300.0,        # K
    T_fac  = 2800.0,        # K
    # Si usaras 'one_spot', usarías f_het y T_het en su lugar.
)

# --- Planeta ---
planet_kwargs = dict(
    planet_name = 'TRAPPIST-1e_like',
    R_p = 0.92*6.371e6,     # m
    gravity = 9.8,          # m/s^2 (o puedes dar mass/log_g)
    T_eq = 250.0,           # K (aprox, solo para escalas)
    d = 12.1*3.086e16       # distancia (m) ~12.1 pc
)

# --- Química (variable) ---
# bulk_species: 'H2'+'He' o un gas principal; param_species: trazas con mixing ratio libre
bulk_species   = ['N2']     # Ej.: ['H2','He'] o ['N2'] para atmósfera tipo terrestre
param_species  = ['H2O','CO2','CH4','O3']  # ← Ajústalo libremente

# Rango de mixing ratios (prior tipo "CLR" o uniforme en log); se define más abajo.
mixing_bounds = {
    'H2O':  (1e-12, 1e-2),
    'CO2':  (1e-12, 1e-1),
    'CH4':  (1e-12, 1e-2),
    'O3':   (1e-12, 1e-4),
}

# --- Perfil P–T y químico ---
PT_profile = 'isotherm'   # 'isotherm', 'gradient', etc.
X_profile  = 'isochem'    # 'isochem' (constante con altura) para empezar

# --- Rejilla espectral ---
wl_min, wl_max, R = 0.6, 5.4, 300    # μm, μm, resolución espectral


In [6]:
# ──────────────────────────────────────────────────────────────────────────────
# 4) Construir estrella, planeta y modelo; leer opacidades
# ──────────────────────────────────────────────────────────────────────────────

# 4.1) Rejilla espectral
wl_model = wl_grid_constant_R(wl_min, wl_max, R)

# 4.2) Estrella con contaminación
star = create_star(
    R_s=star_kwargs['R_s'], T_eff=star_kwargs['T_eff'], log_g=star_kwargs['log_g'], Met=star_kwargs['Met'],
    stellar_grid=star_kwargs['stellar_grid'],
    stellar_contam=star_kwargs['stellar_contam'],
    f_spot=star_kwargs.get('f_spot'), f_fac=star_kwargs.get('f_fac'),
    T_spot=star_kwargs.get('T_spot'), T_fac=star_kwargs.get('T_fac'),
    wl=wl_model
)

# 4.3) Planeta y modelo directo
planet = create_planet(**planet_kwargs)

model = define_model(
    model_name='chem_var_stellar_contam',
    bulk_species=bulk_species,
    param_species=param_species,
    object_type='transiting',
    PT_profile=PT_profile,
    X_profile=X_profile,
    stellar_contam=star_kwargs['stellar_contam'],   # ← activa el término de contaminación en el modelo
)

# 4.4) Cargar opacidades
opac = read_opacities(model, wl_model)
print('Listo: wl grid, star, planet, model y opacidades.')


ParameterOutOfBounds: Parameter 'Teff' exceeds data. Min allowed=3500.000000, entered=2550.000000.

In [7]:
# ──────────────────────────────────────────────────────────────────────────────
# 5) Funciones auxiliares para química y espectro
# ──────────────────────────────────────────────────────────────────────────────

def make_atm_with_chem(planet, model, opac, wl, X_isochem):
    # X_isochem: dict con mixing ratios (fracciones molares) para cada especie en 'param_species'.
    # Presión de referencia y Rp_ref pueden ser valores típicos
    P = np.logspace(2, -6, 80)           # bar → (POSEIDON interpreta internamente unidades)
    P_ref = 1.0                          # bar
    R_p_ref = planet_kwargs['R_p']       # m

    # Construye la atmósfera isóquímica asignando las mezclas
    atmosphere = make_atmosphere(
        planet, model, P=P, P_ref=P_ref, R_p_ref=R_p_ref,
        X_isochem=X_isochem
    )
    # Espectro planetario puro (sin contaminación)
    spec = compute_spectrum(planet, star, model, atmosphere, opac, wl)
    # Factor de contaminación (epsilon) multiplicativo en transmisión
    eps = stellar_contamination(star, wl)
    spec_contam = spec * eps
    return spec, spec_contam, eps

def loguniform(low, high, size=None, rng=rng):
    return 10.0**(rng.uniform(np.log10(low), np.log10(high), size=size))

print("Funciones listas.")


Funciones listas.


In [8]:
# ──────────────────────────────────────────────────────────────────────────────
# 6) Primer experimento: fija química y calcula espectro
# ──────────────────────────────────────────────────────────────────────────────

# Define una química base (puedes ajustarla fija o muestrear)
X0 = {}
for sp in param_species:
    lo, hi = mixing_bounds.get(sp, (1e-12, 1e-2))
    # ejemplo: toma el geométrico medio del rango como punto base
    X0[sp] = float(np.sqrt(lo*hi))

spec_noCont, spec_withCont, eps = make_atm_with_chem(planet, model, opac, wl_model, X0)

plt.figure(figsize=(8,4))
plt.plot(wl_model, spec_noCont, label='Sin contaminación')
plt.plot(wl_model, spec_withCont, label='Con contaminación')
plt.xlabel('Longitud de onda (μm)')
plt.ylabel('Profundidad de tránsito (Rp/Rs)^2')
plt.legend()
plt.title('Efecto de contaminación estelar (two_spots)')
plt.show()


NameError: name 'planet' is not defined

In [9]:
# ──────────────────────────────────────────────────────────────────────────────
# 7) Barrido rápido de química (comparación de espectros)
# ──────────────────────────────────────────────────────────────────────────────

n_cases = 4
spectra = []
labels  = []

for i in range(n_cases):
    Xi = {}
    for sp in param_species:
        lo, hi = mixing_bounds.get(sp, (1e-12, 1e-2))
        Xi[sp] = float(loguniform(lo, hi))
    spec_nc, spec_wc, eps_i = make_atm_with_chem(planet, model, opac, wl_model, Xi)
    spectra.append((Xi, spec_wc))
    labels.append(f"caso {i+1}")

plt.figure(figsize=(9,4))
for (Xi, si), lab in zip(spectra, labels):
    plt.plot(wl_model, si, label=lab)
plt.xlabel('Longitud de onda (μm)')
plt.ylabel('Profundidad de tránsito (Rp/Rs)^2 (con contaminación)')
plt.title('Química variable (mezclas aleatorias)')
plt.legend()
plt.show()

# Muestra los X usados
import pandas as _pd
_pd.DataFrame([x for x,_ in spectra], index=labels)


NameError: name 'planet' is not defined

In [10]:
# ──────────────────────────────────────────────────────────────────────────────
# 8) Barrido 1D sobre una especie (manteniendo las demás fijas)
# ──────────────────────────────────────────────────────────────────────────────

target_species = 'H2O'  # ← cámbialo
grid_vals = np.logspace(-10, -3, 6)

X_base = X0.copy()
curves = []

for xv in grid_vals:
    Xb = X_base.copy()
    Xb[target_species] = float(xv)
    _, spec_wc, _ = make_atm_with_chem(planet, model, opac, wl_model, Xb)
    curves.append((xv, spec_wc))

plt.figure(figsize=(9,4))
for xv, si in curves:
    plt.plot(wl_model, si, label=f'{target_species}={xv:.1e}')
plt.xlabel('Longitud de onda (μm)')
plt.ylabel('Profundidad de tránsito (Rp/Rs)^2 (con contaminación)')
plt.title(f'Sensibilidad a {target_species}')
plt.legend()
plt.show()


NameError: name 'planet' is not defined

In [11]:
# ──────────────────────────────────────────────────────────────────────────────
# 9) (Opcional) Precomputar espectros estelares para el rango de priors
# ──────────────────────────────────────────────────────────────────────────────
# Esto acelera retrievals con contaminación; define un rango razonable
# de T_phot/logg y de T_spot/T_fac consistente con tus priors.

prior_types_hint  = {}
prior_ranges_hint = {}

# (Ejemplo mínimo para ilustrar la llamada; ajusta a tus priors reales)
# NOTA: set_priors formal se hace más abajo; aquí sólo mostramos cómo precalcular:
try:
    _ = precompute_stellar_spectra(
        comm=None, wl_out=wl_model, star=star,
        prior_types=prior_types_hint, prior_ranges=prior_ranges_hint,
        stellar_contam=star_kwargs['stellar_contam'],
        T_step_interp=20, log_g_step_interp=0.1, interp_backend='pysynphot'
    )
    print("Precómputo estelar completado (si estaba disponible).")
except Exception as e:
    print("Aviso: no se precalculó (ajusta prior_types/prior_ranges o ignora si no es necesario).")
    print("Detalle:", repr(e))


Aviso: no se precalculó (ajusta prior_types/prior_ranges o ignora si no es necesario).
Detalle: NameError("name 'star' is not defined")


In [12]:
# ──────────────────────────────────────────────────────────────────────────────
# 10) (Opcional) Simular datos instrumentales binned (placeholder)
# ──────────────────────────────────────────────────────────────────────────────
# Si ya tienes datos, usa load_data con tus CSV; si no, aquí simulamos un dataset.

# Simula errores fraccionales
err_frac = 0.02  # 2% como ejemplo
y = spec_withCont
yerr = np.abs(y)*err_frac
yobs = y + rng.normal(0, yerr)

# Guardamos un CSV simple [wl, bin_width, yobs, yerr] que POSEIDON puede leer
bin_width = np.gradient(wl_model)  # aprox, mitad/ancho real depende del instrumento
df = pd.DataFrame({
    'wl_micron': wl_model,
    'bin_width_micron': bin_width,
    'spectrum': yobs,
    'spectrum_err': yerr
})
csv_path = 'simulated_dataset.csv'
df.to_csv(csv_path, index=False)
print("Escribí datos simulados en:", csv_path)

# Cargar al formato POSEIDON
data = load_data(
    data_dir='.', datasets=['simulated_dataset.csv'], instruments=['SIMULATED'],
    wl_model=wl_model, wl_unit='micron', bin_width='half', spectrum_unit='(Rp/Rs)^2'
)
print("Data cargada para retrieval simulado.")


NameError: name 'spec_withCont' is not defined

In [13]:
# ──────────────────────────────────────────────────────────────────────────────
# 11) Construir priors para química y parámetros base
# ──────────────────────────────────────────────────────────────────────────────

def build_default_priors(planet, star, model, data, mixing_bounds):
    # Prior types: 'uniform', 'gaussian', 'CLR' (para mezclas), etc.
    prior_types = {}
    prior_ranges = {}

    # Química: usa 'CLR' (log-uniform implícito) en los rangos dados
    for sp in model['param_species']:
        lo, hi = mixing_bounds.get(sp, (1e-12, 1e-2))
        prior_types[f'log_{sp}']  = 'CLR'  # Convención de POSEIDON para mezclas
        prior_ranges[f'log_{sp}'] = [lo]   # CLR usa 'low' (límite inferior); high va en config interna

    # Radio/Presión de referencia (puedes fijarlos o darles prior):
    prior_types['R_p_ref']   = 'uniform'
    prior_ranges['R_p_ref']  = [planet['R_p']*0.95, planet['R_p']*1.05]

    prior_types['P_ref']     = 'uniform'
    prior_ranges['P_ref']    = [1e-2, 10.0]  # bar, ejemplo amplio

    # Temperatura isoterma
    prior_types['T_iso']     = 'uniform'
    prior_ranges['T_iso']    = [150.0, 800.0]

    # Contaminación estelar (ajusta a tu caso; aquí un ejemplo amplio)
    if model.get('stellar_contam'):
        if 'two_spots' in model['stellar_contam']:
            prior_types['f_spot'] = 'uniform'; prior_ranges['f_spot'] = [0.0, 0.1]
            prior_types['f_fac']  = 'uniform'; prior_ranges['f_fac']  = [0.0, 0.1]
            prior_types['T_spot'] = 'uniform'; prior_ranges['T_spot'] = [1800.0, 2900.0]
            prior_types['T_fac']  = 'uniform'; prior_ranges['T_fac']  = [2400.0, 3200.0]
        else:
            prior_types['f_het']  = 'uniform'; prior_ranges['f_het']  = [0.0, 0.2]
            prior_types['T_het']  = 'uniform'; prior_ranges['T_het']  = [1800.0, 3200.0]

    # Construye estructura final según POSEIDON
    priors = set_priors(planet, star, model, data, prior_types=prior_types, prior_ranges=prior_ranges)
    return priors

priors = build_default_priors(planet, star, model, data, mixing_bounds)
print("Priors listos.")


NameError: name 'planet' is not defined

In [14]:
# ──────────────────────────────────────────────────────────────────────────────
# 12) (Opcional) Retrieval (plantilla)
# ──────────────────────────────────────────────────────────────────────────────
RUN_RETRIEVAL = False  # ← pon True si ya tienes poseidon y un sampler instalado

if RUN_RETRIEVAL:
    try:
        # Algunos builds de POSEIDON exponen un wrapper general:
        from POSEIDON.retrieval import run_retrieval
        results = run_retrieval(planet=planet, star=star, model=model, data=data, priors=priors,
                                sampler='dynesty', nlive=400, dlogz=0.05)
        print("Retrieval terminado (wrapper run_retrieval).")
    except Exception as e:
        print("Wrapper run_retrieval no disponible o falló. Probando interfaces específicas...")
        try:
            # Interfaces específicas (pueden variar con la versión):
            from POSEIDON.retrieval import Dynesty_retrieval
            results = Dynesty_retrieval(planet, star, model, data, priors,
                                        nlive=400, dlogz=0.05)
            print("Retrieval terminado (Dynesty_retrieval).")
        except Exception as e2:
            print("Retrieval no ejecutado. Revisa instalación de backends o usa tu flujo preferido.")
            print("Error 1:", repr(e))
            print("Error 2:", repr(e2))
else:
    print("Retrieval desactivado. Actívalo cuando tengas el backend de muestreo listo (dynesty/emcee/pymultinest).")


Retrieval desactivado. Actívalo cuando tengas el backend de muestreo listo (dynesty/emcee/pymultinest).


In [15]:
# ──────────────────────────────────────────────────────────────────────────────
# 13) Batch de química desde una tabla (CSV/Dict) y exportar espectros
# ──────────────────────────────────────────────────────────────────────────────
# Formato esperado: columnas con nombres de especies paramétricas y filas con casos.

cases = pd.DataFrame([
    {'H2O':1e-4,'CO2':1e-3,'CH4':1e-6,'O3':1e-8},
    {'H2O':1e-5,'CO2':1e-4,'CH4':1e-5,'O3':1e-7},
    {'H2O':1e-6,'CO2':1e-5,'CH4':1e-4,'O3':1e-9},
])

all_specs = []
for idx,row in cases.iterrows():
    Xi = {sp: float(row.get(sp, 1e-6)) for sp in param_species}
    _, spec_wc, _ = make_atm_with_chem(planet, model, opac, wl_model, Xi)
    all_specs.append(spec_wc)

all_specs = np.array(all_specs)
out = pd.DataFrame(all_specs, columns=[f"{w:.5f}" for w in wl_model])
out.insert(0, 'case_id', np.arange(len(cases)))
out_path = 'chemistry_sweep_spectra.csv'
out.to_csv(out_path, index=False)
print(f"Exportado: {out_path}")


NameError: name 'planet' is not defined